# Vývoj aplikácií na generovanie obrázkov (OpenAI)

LLM nie sú len o generovaní textu. Môžete tiež generovať obrázky z textových popisov. Obrázky ako modalita sú užitočné v oblasti zdravotníckych technológií, architektúry, turizmu, vývoja hier, marketingu a ďalších. V tejto lekcii pracujeme s dnešnými modelmi **GPT Image** na platforme OpenAI.

## Ciele učenia

Na konci tejto lekcie budete vedieť:

- Vysvetliť, čo je generovanie obrázkov a kde je užitočné.
- Pochopiť rodinu modelov `gpt-image` a ako sa líši od starších modelov DALL·E.
- Vytvoriť aplikáciu na generovanie obrázkov a upravovať obrázky.

## Čo je generovanie obrázkov?

Modely na generovanie obrázkov vytvárajú obrázky na základe textového podnetu. Moderné modely, ako `gpt-image`, sa počas tréningu učia vzťah medzi textom a obrázkami, potom postupne premieňajú náhodný šum na obrázok, ktorý zodpovedá vášmu podnetu.

Dve známe rodiny modelov na generovanie obrázkov sú:

- **`gpt-image` (OpenAI)** — aktuálna generácia použitá v tejto lekcii. Podporuje generovanie obrázkov z textu a úpravu obrázkov (inpainting s maskou).
- **Midjourney** — populárny model tretej strany so svojou vlastnou službou a pracovným tokom založeným na Discorde.

> Staršie modely obrázkov OpenAI — **DALL·E 2** a **DALL·E 3** — sú zastarané. DALL·E 3 už nie je dostupný pre nové nasadenia a funkcie ako `create_variation` existovali iba v DALL·E 2. Na nové aplikácie používajte modely `gpt-image`.

> **Dôležité:** Modely `gpt-image` vracajú vytvorený obrázok ako **base64** (`b64_json`), nie ako URL. Váš kód dekóduje base64 reťazec na bajty a ukladá ho — neexistuje žiadne URL obrázku na stiahnutie.


## Vytvorenie vašej prvej aplikácie na generovanie obrázkov

Čo všetko teda potrebujete na vytvorenie aplikácie na generovanie obrázkov? Potrebujete nasledujúce knižnice:

- **python-dotenv**, túto knižnicu sa vám veľmi odporúča použiť na uchovávanie vašich tajomstiev v súbore *.env* mimo kódu.
- **openai**, túto knižnicu použijete na interakciu s OpenAI API.
- **pillow**, na prácu s obrázkami v Pythone.


1. Vytvorte súbor *.env* s nasledujúcim obsahom:

    ```text
    OPENAI_API_KEY='<add your OpenAI key here>'
    ```



1. Zhromaždite vyššie uvedené knižnice do súboru nazvaného *requirements.txt* nasledovne:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Ďalej vytvorte virtuálne prostredie a nainštalujte knižnice:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> Pre Windows použite na vytvorenie a aktiváciu virtuálneho prostredia nasledujúce príkazy:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Pridajte nasledujúci kód do súboru s názvom *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai

    # import dotenv
    dotenv.load_dotenv()

    # Vytvorte objekt OpenAI (číta OPENAI_API_KEY z vášho .env)
    client = openai.OpenAI()


    try:
        # Vytvorte obrázok pomocou API na generovanie obrázkov
        generation_response = client.images.generate(
            model="gpt-image-1",
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Zadajte text vášho promptu sem
            size='1024x1024',
            n=1
        )
        # Nastavte adresár pre uložený obrázok
        image_dir = os.path.join(os.curdir, 'images')

        # Ak adresár neexistuje, vytvorte ho
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Inicializujte cestu k obrázku (dávajte pozor, aby bol typ súboru png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # modely gpt-image vracajú obrázok ako base64 (b64_json), nie ako URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Zobrazte obrázok v predvolenom prehliadači obrázkov
        image = Image.open(image_path)
        image.show()

    # zachytiť výnimky
    except openai.BadRequestError as err:
        print(err)

    ```

Poďme si vysvetliť tento kód:

- Najskôr importujeme knižnice, ktoré potrebujeme, vrátane knižnice OpenAI, knižnice dotenv, modulu base64 a knižnice Pillow.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai
    ```

- Potom vytvoríme klienta, ktorý načíta API kľúč z vášho súboru ``.env``.

    ```python
    # Vytvoriť objekt OpenAI
    client = openai.OpenAI()
    ```

- Ďalej generujeme obrázok:

    ```python
    generation_response = client.images.generate(
        model="gpt-image-1",
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Zadajte svoj text výzvy sem
        size='1024x1024',
        n=1
    )
    ```

    Modely `gpt-image` vrátia obrázok ako **base64** reťazec v `data[0].b64_json`. Dekódujeme ho na bajty a zapíšeme do súboru — neexistuje žiadna URL na stiahnutie.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Nakoniec otvoríme obrázok a použijeme štandardný prehliadač obrázkov na jeho zobrazenie:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Viac detailov o generovaní obrázkov

Pozrime sa na parametre `images.generate`:

- **model**, je model obrázku, ktorý sa použije (napríklad `gpt-image-1`).
- **prompt**, je textový popis použitý na generovanie obrázku. Tu je to "Zajko na koni, drží lízatko, na hmlistej lúke, kde rastú narcisky".
- **size**, je veľkosť generovaného obrázku (napríklad `1024x1024`, `1536x1024`, `1024x1536` alebo `"auto"`).
- **n**, je počet generovaných obrázkov. Tu generujeme jeden.

> Modely obrázkov neberú parameter `temperature` — to je kontrola generovania textu. Ak chcete rozmanitosť, zavolajte API znova; ak chcete znížiť rozmanitosť, spravte svoj opis presnejším.

## Ďalšie možnosti generovania obrázkov

Videli ste, ako vygenerovať obrázok s pár riadkami Pythonu. Modely `gpt-image` môžu tiež **editovať** existujúci obrázok. Poskytnutím obrázku, voliteľnej **masky** (ktorá označuje oblasť na zmenu) a popisu môžete upraviť časť obrázku — napríklad pridať klobúk nášmu zajkovi.

```python
response = client.images.edit(
  model="gpt-image-1",
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# úpravy sú tiež vrátené ako base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

Základný obrázok obsahuje iba zajaca; finálny obrázok má klobúk na zajacovi.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vyhlásenie o zodpovednosti**:
Tento dokument bol preložený pomocou AI prekladateľskej služby [Co-op Translator](https://github.com/Azure/co-op-translator). Hoci sa snažíme o presnosť, vezmite prosím na vedomie, že automatické preklady môžu obsahovať chyby alebo nepresnosti. Pôvodný dokument v jeho natívnom jazyku by mal byť považovaný za autoritatívny zdroj. Pre kritické informácie sa odporúča profesionálny ľudský preklad. Nie sme zodpovední za žiadne nedorozumenia alebo nesprávne interpretácie vyplývajúce z použitia tohto prekladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
